In [19]:
from pathlib import Path
from tensorboard.backend.event_processing import event_accumulator
import pandas as pd


def export_tb_log_to_csv_grouped(log_dir: str, tags: list[str], out_csv: str):
    log_path = Path(log_dir)
    if not log_path.exists():
        raise FileNotFoundError(f"Log directory not found: {log_dir}")

    ea = event_accumulator.EventAccumulator(str(log_path))
    ea.Reload()

    available_tags = ea.Tags().get("scalars", [])
    dfs = []

    for tag in tags:
        if tag not in available_tags:
            print(f"[Warning] Tag not found in {log_dir}: {tag}")
            continue

        events = ea.Scalars(tag)
        if len(events) == 0:
            print(f"[Warning] Empty tag in {log_dir}: {tag}")
            continue

        # One tag -> one 3-column dataframe
        df_tag = pd.DataFrame({
            "step": [e.step for e in events],
            "wall_time": [e.wall_time for e in events],
            "val": [e.value for e in events],
        }).reset_index(drop=True)

        # Convert columns to two-level header: (tag, step), (tag, wall_time), (tag, val)
        df_tag.columns = pd.MultiIndex.from_product([[tag], df_tag.columns])

        dfs.append(df_tag)

    if len(dfs) == 0:
        raise ValueError(f"No valid scalar tags found in {log_dir}")

    # Horizontal concat by row number, NOT by step or wall_time
    df_out = pd.concat(dfs, axis=1)

    # Save exactly as two header rows
    df_out.to_csv(out_csv, index=False)
    print(f"[Saved] {out_csv}")

    return df_out


def export_multiple_logs(log_dirs: list[str], tags: list[str], out_dir: str):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for log_dir in log_dirs:
        log_path = Path(log_dir)
        out_csv = out_dir / f"{log_path.name}.csv"
        try:
            export_tb_log_to_csv_grouped(str(log_path), tags, str(out_csv))
        except Exception as e:
            print(f"[Failed] {log_dir}: {e}")



In [20]:
log_dirs = [
        "/home/shahao/projects/DECODE_Internal/log/09-43-23/0",
        "/home/shahao/projects/DECODE_Internal/log/09-43-23/1",
    ]

tags = [
    "eval/effcy_lat",
    "eval/jac",
    "eval/rmse_lat",
    "eval/rmse_ax",
]

out_dir = "/home/shahao/projects/DECODE_Internal/Manuscript/plex/warm_start"

export_multiple_logs(log_dirs, tags, out_dir)

[Saved] /home/shahao/projects/DECODE_Internal/Manuscript/plex/warm_start/0.csv
[Saved] /home/shahao/projects/DECODE_Internal/Manuscript/plex/warm_start/1.csv
